# Лабораторная работа 2. Продолжение NumPy
Михайлов Яков Витальевич, группа 1521741.

Задачи 6–8 из «Тема 1.1. NumPy». Запускайте ячейки сверху вниз.
Размеры массивов сохранены из условия. Задача 7.4 требует примерно 1–1.5 ГБ памяти; циклы могут выполняться десятки секунд.
Замеры ниже получены при выполнении блокнота и зависят от оборудования.

In [1]:
from pathlib import Path
import gc
import timeit
import platform
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from threadpoolctl import threadpool_limits

rng = np.random.default_rng(42)
LAB_DIR = Path.cwd()
if (LAB_DIR / "labs/02_numpy_pandas").is_dir():
    LAB_DIR = LAB_DIR / "labs/02_numpy_pandas"

def measure(function, repeats=3, number=1):
    return float(np.median(timeit.repeat(function, repeat=repeats, number=number))) / number

print("Python:", platform.python_version())
print("NumPy:", np.__version__, "Pandas:", pd.__version__, "SciPy:", scipy.__version__)

Python: 3.12.14
NumPy: 2.3.5 Pandas: 2.2.3 SciPy: 1.17.0


## 6. Индексация и срезы
### 6.1
Границы 5 и 15 трактуем как индексы: выбираем arr[5:15], затем переворачиваем.
Если считать именно порядковые номера элементов, соответствующий срез arr[4:14][::-1] приведён отдельно.

In [2]:
arr = np.arange(20)
print("Каждый третий с индекса 1:", arr[1::3])
print("Индексы от 5 до 15, без 15, в обратном порядке:", arr[5:15][::-1])
print("Порядковые номера от 5 до 15, без 15:", arr[4:14][::-1])
print("Последние пять наоборот:", arr[-5:][::-1])
np.testing.assert_array_equal(arr[5:15][::-1], np.arange(14, 4, -1))

Каждый третий с индекса 1: [ 1  4  7 10 13 16 19]
Индексы от 5 до 15, без 15, в обратном порядке: [14 13 12 11 10  9  8  7  6  5]
Порядковые номера от 5 до 15, без 15: [13 12 11 10  9  8  7  6  5  4]
Последние пять наоборот: [19 18 17 16 15]


### 6.2
Номера строк 2–4 и столбцов 1–3 считаем индексами, включая обе границы. Шахматная выборка содержит 18 элементов.

In [3]:
matrix = rng.integers(0, 10, size=(6, 6))
rows, cols = np.indices(matrix.shape)
mask = (rows + cols) % 2 == 0
print("Матрица:\n", matrix)
print("Подматрица по индексам:\n", matrix[2:5, 1:4])
print("Если считать порядковые номера:\n", matrix[1:4, 0:3])
print("Главная диагональ:", np.diagonal(matrix))
print("Шахматная маска:\n", mask.astype(int))
print("Выбранные элементы:", matrix[mask])
assert matrix[mask].size == 18

Матрица:
 [[0 7 6 4 4 8]
 [0 6 2 0 5 9]
 [7 7 7 7 5 1]
 [8 4 5 3 1 9]
 [7 6 4 8 5 4]
 [4 2 0 5 8 0]]
Подматрица по индексам:
 [[7 7 7]
 [4 5 3]
 [6 4 8]]
Если считать порядковые номера:
 [[0 6 2]
 [7 7 7]
 [8 4 5]]
Главная диагональ: [0 6 7 3 5 0]
Шахматная маска:
 [[1 0 1 0 1 0]
 [0 1 0 1 0 1]
 [1 0 1 0 1 0]
 [0 1 0 1 0 1]
 [1 0 1 0 1 0]
 [0 1 0 1 0 1]]
Выбранные элементы: [0 6 4 6 0 9 7 7 5 4 3 9 7 4 5 2 5 0]


### 6.3. Попарные индексы и прямоугольный блок

In [4]:
a = np.array([10, 20, 30, 40, 50, 60])
m = np.arange(12).reshape(3, 4)
print("Выборка a:", a[[0, 2, 4]])
print("Попарная выборка:", m[[0, 1, 2], [1, 3, 0]])
block = m[np.ix_([0, 2], [1, 3])]
print("Блок 2 × 2:\n", block)
np.testing.assert_array_equal(block, [[1, 3], [9, 11]])

Выборка a: [10 30 50]
Попарная выборка: [1 7 8]
Блок 2 × 2:
 [[ 1  3]
 [ 9 11]]


### 6.4. Логические маски
Каждый подпункт выполняем независимо на исходном x. Иначе после замены отрицательных значений на ноль смены знака исчезнут.
Разные знаки означают строго положительного и строго отрицательного соседа; ноль не относится ни к одному из них. Крайние элементы без двух соседей не меняем.

In [5]:
np.random.seed(42)
x = np.random.randn(1000)
print("Количество > 1:", np.count_nonzero(x > 1))
nonnegative = x.copy()
nonnegative[nonnegative < 0] = 0

def zero_between_opposite_neighbors(values):
    result = values.copy()
    positions = np.zeros(values.shape, dtype=bool)
    positions[1:-1] = ((values[:-2] < 0) & (values[2:] > 0)) | ((values[:-2] > 0) & (values[2:] < 0))
    result[positions] = 0
    return result, positions

changed, positions = zero_between_opposite_neighbors(x)
print("Первые 12 исходных:", x[:12])
print("После замены отрицательных:", nonnegative[:12])
print("После обработки соседей:", changed[:12])
print("Позиций с противоположными знаками соседей:", positions.sum())
np.testing.assert_array_equal(zero_between_opposite_neighbors(np.array([-1., 5., 2., -3., 0.]))[0], [-1, 0, 0, -3, 0])
assert np.all(nonnegative >= 0) and changed[0] == x[0] and changed[-1] == x[-1]

Количество > 1: 155
Первые 12 исходных: [ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337 -0.23413696
  1.57921282  0.76743473 -0.46947439  0.54256004 -0.46341769 -0.46572975]
После замены отрицательных: [0.49671415 0.         0.64768854 1.52302986 0.         0.
 1.57921282 0.76743473 0.         0.54256004 0.         0.        ]
После обработки соседей: [ 0.49671415 -0.1382643   0.          0.          0.          0.
  0.          0.         -0.46947439  0.54256004  0.          0.        ]
Позиций с противоположными знаками соседей: 512


## 7. Memory-mapped файлы и скорость вычислений
### 7.1. Файл big.dat

In [6]:
data_dir = LAB_DIR / "data"
data_dir.mkdir(exist_ok=True)
path = data_dir / "big.dat"
shape = (50000, 200)
batch_rows = 1000
mapped = np.memmap(path, mode="w+", dtype=np.float32, shape=shape)
reference_min, reference_max, reference_sum = np.inf, -np.inf, 0.0
for start in range(0, shape[0], batch_rows):
    stop = min(start + batch_rows, shape[0])
    batch = rng.random((stop - start, shape[1]), dtype=np.float32)
    mapped[start:stop] = batch
    reference_min = min(reference_min, float(batch.min()))
    reference_max = max(reference_max, float(batch.max()))
    reference_sum += batch.sum(dtype=np.float64)
mapped.flush()
del mapped, batch
expected_bytes = 50000 * 200 * 4
print("Ожидаемый размер:", expected_bytes, "байт")
print("Фактический размер:", path.stat().st_size, "байт")
assert path.stat().st_size == expected_bytes

Ожидаемый размер: 40000000 байт
Фактический размер: 40000000 байт


### 7.2. Чтение батчами
Читаем не более 1000 строк за шаг. Для суммы используем float64, чтобы уменьшить накопление ошибки. Среднее — общая сумма, делённая на общее число элементов.
Полный срез mapped[:] не используется. Сам memmap-срез является представлением; страницы подгружаются ОС по мере обращения.

In [7]:
mapped = np.memmap(path, mode="r", dtype=np.float32, shape=shape)
global_min, global_max, total, count = np.inf, -np.inf, 0.0, 0
for start in range(0, shape[0], batch_rows):
    batch = mapped[start:min(start + batch_rows, shape[0])]
    global_min = min(global_min, float(batch.min()))
    global_max = max(global_max, float(batch.max()))
    total += batch.sum(dtype=np.float64)
    count += batch.size
mean = total / count
print("min =", global_min, "max =", global_max, "mean =", mean)
assert count == 10_000_000
assert global_min == reference_min and global_max == reference_max
np.testing.assert_allclose(mean, reference_sum / count, rtol=1e-14)
del batch, mapped

min = 0.0 max = 0.9999998807907104 mean = 0.49993614274504183


### 7.3. Чистые Python-циклы, C-order и F-order
Во всех случаях одинаковые значения и доступ arr[i, j]. Создание массивов не входит в замер.

In [8]:
def row_sum(arr):
    total = 0.0
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            total += arr[i, j]
    return total

def column_sum(arr):
    total = 0.0
    for j in range(arr.shape[1]):
        for i in range(arr.shape[0]):
            total += arr[i, j]
    return total

c_array = rng.random((3000, 3000))
f_array = np.array(c_array, order="F")
reference = c_array.sum()
records = []
for order, array in [("C", c_array), ("F", f_array)]:
    for traversal, function in [("по строкам", row_sum), ("по столбцам", column_sum)]:
        np.testing.assert_allclose(function(array), reference, rtol=1e-10)
        elapsed = measure(lambda: function(array))
        records.append({"Порядок памяти": order, "Обход": traversal, "Время, с": elapsed})
print(pd.DataFrame(records).to_string(index=False))
del c_array, f_array, array
_ = gc.collect()

Порядок памяти       Обход  Время, с
             C  по строкам  1.196867
             C по столбцам  1.233236
             F  по строкам  1.259727
             F по столбцам  1.218811


В C-order непрерывно расположены строки, в F-order — столбцы. Обход вдоль непрерывного направления обычно лучше использует кэш. Накладные расходы Python могут уменьшать наблюдаемую разницу.

### 7.4. Выражение на 10 млн элементов float64
Одинаковые данные для трёх способов. Берём a в [1, 2), b в [0, 1), поэтому знаменатель не равен нулю.
Вариант out= использует два заранее выделенных буфера. Создание входов, списков и буферов не входит в замер; выделение результата внутри обычных выражений входит.

In [9]:
N = 10_000_000
a = rng.random(N) + 1.0
b = rng.random(N)
a_list, b_list = a.tolist(), b.tolist()

def list_expression():
    result = []
    for ai, bi in zip(a_list, b_list):
        result.append((ai + bi) / (ai - bi + 1.0))
    return result

list_seconds = measure(list_expression)
list_result = list_expression()
# Проверяем весь результат частями, без дополнительного полного массива.
for start in range(0, N, 100000):
    end = min(start + 100000, N)
    np.testing.assert_allclose(list_result[start:end], (a[start:end] + b[start:end]) / (a[start:end] - b[start:end] + 1))
del list_result, a_list, b_list
_ = gc.collect()

def numpy_expression():
    return (a + b) / (a - b + 1)

numpy_seconds = measure(numpy_expression)
result_buffer = np.empty_like(a)
denominator = np.empty_like(a)

def buffer_expression():
    np.add(a, b, out=result_buffer)
    np.subtract(a, b, out=denominator)
    np.add(denominator, 1, out=denominator)
    np.divide(result_buffer, denominator, out=result_buffer)
    return result_buffer

buffer_seconds = measure(buffer_expression)
np.testing.assert_allclose(numpy_expression(), buffer_expression())
print(pd.DataFrame({"Способ": ["Python list", "NumPy", "NumPy out="],
                    "Медиана, с": [list_seconds, numpy_seconds, buffer_seconds]}).to_string(index=False))
del a, b, result_buffer, denominator
_ = gc.collect()

     Способ  Медиана, с
Python list    1.002843
      NumPy    0.092988
 NumPy out=    0.066145


NumPy выполняет операции над массивами в скомпилированном коде. out= уменьшает количество выделений временных массивов, но не объединяет четыре прохода в один и не гарантирует ускорение на любой машине.

## 8. Разреженные матрицы
### 8.1. COO → CSR и совпадающие координаты
Конкретные списки row, col, data в материалах отсутствуют: создаём воспроизводимый пример длиной 10 000 и намеренно повторяем 100 координат. Значения положительны, поэтому при суммировании дубликатов не возникают нули.

In [10]:
row = rng.integers(0, 5000, 10000).tolist()
col = rng.integers(0, 5000, 10000).tolist()
data = rng.random(10000).tolist()
row[-100:], col[-100:] = row[:100], col[:100]
coo = sparse.coo_matrix((data, (row, col)), shape=(5000, 5000))
csr = coo.tocsr()
unique_pairs = len(set(zip(row, col)))
print("COO nnz до объединения:", coo.nnz)
print("Уникальных координат:", unique_pairs)
print("CSR nnz:", csr.nnz)
assert coo.nnz == 10000 and csr.nnz == unique_pairs
np.testing.assert_allclose(csr.sum(), sum(data))
# Проверка суммирования значений при повторении одной координаты.
key = (row[0], col[0])
expected = sum(v for i, j, v in zip(row, col, data) if (i, j) == key)
np.testing.assert_allclose(csr[key], expected)

COO nnz до объединения: 10000
Уникальных координат: 9898
CSR nnz: 9898


nnz считает хранимые значения, включая явно записанные нули. Для данных с взаимным сокращением значений после суммирования понадобился бы eliminate_zeros().

### 8.2. Размер 4000 × 4000, плотность 0.5%
Сравниваем память числовых буферов без небольших накладных расходов Python-объектов. Для воспроизводимого сравнения ограничиваем BLAS одним потоком только внутри замеров.

In [11]:
def csr_bytes(matrix):
    return matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes

A = sparse.random(4000, 4000, density=0.005, format="csr", dtype=np.float64, random_state=42)
dense = A.toarray()
vector = rng.random(4000)
with threadpool_limits(limits=1):
    np.testing.assert_allclose(A @ vector, dense @ vector, rtol=1e-12, atol=1e-12)
    t_sparse = measure(lambda: A @ vector, number=10)
    t_dense = measure(lambda: dense @ vector, number=10)
print(pd.DataFrame({"Формат": ["dense", "CSR"], "Байты": [dense.nbytes, csr_bytes(A)],
                    "Время, с": [t_dense, t_sparse]}).to_string(index=False))
assert A.nnz == 80000 and dense.nbytes == 128000000
del A, dense, vector
_ = gc.collect()

Формат     Байты  Время, с
 dense 128000000  0.006381
   CSR    976004  0.000097


### 8.3. Зависимость от плотности
Таблица вместо графика разрешена условием. Для каждой плотности обе формы представляют одну матрицу; преобразования исключены из замера. Указываем первый измеренный проигрыш sparse и интервал перед ним, а не универсальную точную границу.

In [12]:
densities = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
n = 2000
vector = rng.random(n)
results = []
with threadpool_limits(limits=1):
    for density in densities:
        A = sparse.random(n, n, density=density, format="csr", dtype=np.float64, random_state=42)
        dense = A.toarray()
        np.testing.assert_allclose(A @ vector, dense @ vector, rtol=1e-12, atol=1e-12)
        ts = measure(lambda: A @ vector, repeats=5, number=10)
        td = measure(lambda: dense @ vector, repeats=5, number=10)
        results.append({"Плотность": density, "CSR, мс": ts * 1000, "dense, мс": td * 1000,
                        "dense / CSR": td / ts, "Быстрее": "CSR" if ts < td else "dense"})
        del A, dense
comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))
losing = [i for i, r in enumerate(results) if r["Быстрее"] == "dense"]
if losing:
    i = losing[0]
    print("Первый измеренный проигрыш CSR при плотности", densities[i])
    if i > 0:
        print("Первый переход на исследованной сетке между", densities[i-1], "и", densities[i])
    else:
        print("CSR не выигрывает уже при минимальной проверенной плотности.")
else:
    print("На исследованной сетке CSR выигрывает во всех измерениях; граница не обнаружена.")

 Плотность  CSR, мс  dense, мс  dense / CSR Быстрее
     0.001 0.007159   1.538177   214.859156     CSR
     0.005 0.021045   1.392722    66.178933     CSR
     0.010 0.123121   1.578038    12.816951     CSR
     0.050 0.136805   1.396217    10.205919     CSR
     0.100 0.291760   1.338574     4.587936     CSR
     0.200 0.664591   1.370264     2.061815     CSR
     0.500 1.784112   1.435664     0.804694   dense
     1.000 3.649818   1.529984     0.419195   dense
Первый измеренный проигрыш CSR при плотности 0.5
Первый переход на исследованной сетке между 0.2 и 0.5


**Вывод:** при малой плотности CSR хранит мало значений и экономит память. С ростом плотности расходы на индексы и косвенный доступ становятся существенными. Граница по скорости зависит от оборудования, реализации BLAS и числа потоков; результаты таблицы относятся к данному запуску.